# 02b — Improved Training

Two parts:
1. **Resume EfficientNet-B0 fine-tuning** from the phase-1 checkpoint (backbone was frozen for 3 epochs, now we unfreeze everything and continue with a small LR)
2. **Train a simple custom CNN from scratch** as a baseline comparison model

In [1]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PROCESSED = Path('../data/processed')
WEIGHTS_EFFICIENTNET = Path('../backend/model/weights/efficientnet_b0_finetuned.pth')
WEIGHTS_CNN          = Path('../backend/model/weights/custom_cnn_baseline.pth')
WEIGHTS_EFFICIENTNET.parent.mkdir(parents=True, exist_ok=True)

print('Device:', DEVICE)

Device: cpu


## Dataset & Transforms

In [2]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class AIDetectionDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['filepath']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, int(row['label'])

train_ds = AIDetectionDataset(PROCESSED / 'train.csv', train_transform)
val_ds   = AIDetectionDataset(PROCESSED / 'val.csv',   val_transform)

# Weighted sampler to handle class imbalance
labels_arr = train_ds.df['label'].values
class_counts = np.bincount(labels_arr)
class_weights = 1.0 / class_counts
sample_weights = class_weights[labels_arr]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,   num_workers=0)

print(f'Train: {len(train_ds)} images | Val: {len(val_ds)} images')
print(f'Class counts — REAL: {class_counts[0]}, AI_GENERATED: {class_counts[1]}')

Train: 84000 images | Val: 18000 images
Class counts — REAL: 42000, AI_GENERATED: 42000


---
## Part 1 — Resume EfficientNet-B0 Fine-tuning

The phase-1 checkpoint (3 epochs, frozen backbone) is loaded.
All layers are now unfrozen and trained with a small learning rate (2e-5).

In [3]:
# Build the same architecture as in 02_training.ipynb
eff_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = eff_model.classifier[1].in_features
eff_model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features, 2),
)

# Load phase-1 checkpoint (3 epochs, frozen backbone)
state_dict = torch.load(WEIGHTS_EFFICIENTNET, map_location=DEVICE, weights_only=True)
eff_model.load_state_dict(state_dict)
eff_model = eff_model.to(DEVICE)

# Unfreeze ALL layers for full fine-tuning
for param in eff_model.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in eff_model.parameters())
print(f'Loaded phase-1 checkpoint. Trainable params: {total_params:,}')
print('Backbone unfrozen — starting full fine-tuning from phase-1 weights.')

Loaded phase-1 checkpoint. Trainable params: 4,010,110
Backbone unfrozen — starting full fine-tuning from phase-1 weights.


In [ ]:
EPOCHS_FINETUNE = 12
LR_FINETUNE     = 2e-5   # small LR: backbone already has good ImageNet features
PATIENCE        = 4

criterion   = nn.CrossEntropyLoss()
optimizer   = optim.AdamW(eff_model.parameters(), lr=LR_FINETUNE, weight_decay=1e-4)
scheduler   = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FINETUNE)

best_val_loss  = float('inf')
patience_count = 0
eff_history    = []

for epoch in range(1, EPOCHS_FINETUNE + 1):
    # ---- Train ----
    eff_model.train()
    train_loss, train_correct = 0.0, 0
    for images, labels_batch in train_loader:
        images, labels_batch = images.to(DEVICE), labels_batch.to(DEVICE)
        optimizer.zero_grad()
        outputs = eff_model(images)
        loss    = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels_batch).sum().item()

    # ---- Validate ----
    eff_model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for images, labels_batch in val_loader:
            images, labels_batch = images.to(DEVICE), labels_batch.to(DEVICE)
            outputs   = eff_model(images)
            loss      = criterion(outputs, labels_batch)
            val_loss    += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels_batch).sum().item()

    train_loss /= len(train_ds)
    val_loss   /= len(val_ds)
    train_acc   = train_correct / len(train_ds)
    val_acc     = val_correct   / len(val_ds)
    scheduler.step()

    eff_history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                        'train_acc': train_acc, 'val_acc': val_acc})
    print(f'[EfficientNet] Epoch {epoch:2d} | '
          f'train_loss={train_loss:.4f} acc={train_acc:.4f} | '
          f'val_loss={val_loss:.4f} acc={val_acc:.4f}')

    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        patience_count = 0
        torch.save(eff_model.state_dict(), WEIGHTS_EFFICIENTNET)
        print(f'  -> Saved best EfficientNet (val_loss={val_loss:.4f})')
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nEfficientNet fine-tuning complete. Best val_loss: {best_val_loss:.4f}')
print(f'Weights saved to: {WEIGHTS_EFFICIENTNET}')

---
## Part 2 — Simple Custom CNN (Trained from Scratch)

A lightweight 4-block CNN with no pre-trained weights.
Used as a baseline to compare against EfficientNet.
Architecture: Conv→BN→ReLU→MaxPool ×4, then GlobalAvgPool → Dropout → Linear.

In [ ]:
class SimpleCNN(nn.Module):
    """4-block CNN trained from scratch for AI vs Real image classification."""

    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1 — 224x224 -> 112x112
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2 — 112x112 -> 56x56
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3 — 56x56 -> 28x28
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 4 — 28x28 -> 14x14
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),   # (B, 256, 1, 1)
            nn.Flatten(),              # (B, 256)
            nn.Dropout(p=0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


cnn_model = SimpleCNN(num_classes=2).to(DEVICE)
total_cnn = sum(p.numel() for p in cnn_model.parameters())
print(f'SimpleCNN parameters: {total_cnn:,}  (no pre-trained weights)')

In [ ]:
EPOCHS_CNN = 10
LR_CNN     = 1e-3
PATIENCE_CNN = 4

criterion_cnn = nn.CrossEntropyLoss()
optimizer_cnn = optim.AdamW(cnn_model.parameters(), lr=LR_CNN, weight_decay=1e-4)
scheduler_cnn = optim.lr_scheduler.CosineAnnealingLR(optimizer_cnn, T_max=EPOCHS_CNN)

best_cnn_val_loss  = float('inf')
patience_cnn_count = 0
cnn_history        = []

for epoch in range(1, EPOCHS_CNN + 1):
    # ---- Train ----
    cnn_model.train()
    train_loss, train_correct = 0.0, 0
    for images, labels_batch in train_loader:
        images, labels_batch = images.to(DEVICE), labels_batch.to(DEVICE)
        optimizer_cnn.zero_grad()
        outputs = cnn_model(images)
        loss    = criterion_cnn(outputs, labels_batch)
        loss.backward()
        optimizer_cnn.step()
        train_loss    += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels_batch).sum().item()

    # ---- Validate ----
    cnn_model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for images, labels_batch in val_loader:
            images, labels_batch = images.to(DEVICE), labels_batch.to(DEVICE)
            outputs   = cnn_model(images)
            loss      = criterion_cnn(outputs, labels_batch)
            val_loss    += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels_batch).sum().item()

    train_loss /= len(train_ds)
    val_loss   /= len(val_ds)
    train_acc   = train_correct / len(train_ds)
    val_acc     = val_correct   / len(val_ds)
    scheduler_cnn.step()

    cnn_history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                        'train_acc': train_acc, 'val_acc': val_acc})
    print(f'[SimpleCNN]    Epoch {epoch:2d} | '
          f'train_loss={train_loss:.4f} acc={train_acc:.4f} | '
          f'val_loss={val_loss:.4f} acc={val_acc:.4f}')

    if val_loss < best_cnn_val_loss:
        best_cnn_val_loss  = val_loss
        patience_cnn_count = 0
        torch.save(cnn_model.state_dict(), WEIGHTS_CNN)
        print(f'  -> Saved best CNN (val_loss={val_loss:.4f})')
    else:
        patience_cnn_count += 1
        if patience_cnn_count >= PATIENCE_CNN:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nSimpleCNN training complete. Best val_loss: {best_cnn_val_loss:.4f}')
print(f'Weights saved to: {WEIGHTS_CNN}')

---
## Model Comparison — Training Curves

In [ ]:
eff_df = pd.DataFrame(eff_history)
cnn_df = pd.DataFrame(cnn_history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Validation loss
axes[0].plot(eff_df['epoch'], eff_df['val_loss'], 'b-o', label='EfficientNet-B0 (fine-tuned)')
axes[0].plot(cnn_df['epoch'], cnn_df['val_loss'], 'r-o', label='SimpleCNN (from scratch)')
axes[0].set_title('Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation accuracy
axes[1].plot(eff_df['epoch'], eff_df['val_acc'], 'b-o', label='EfficientNet-B0 (fine-tuned)')
axes[1].plot(cnn_df['epoch'], cnn_df['val_acc'], 'r-o', label='SimpleCNN (from scratch)')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('EfficientNet-B0 vs SimpleCNN — Training Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('model_comparison_curves.png', dpi=150)
plt.show()

print(f'EfficientNet best val_acc: {eff_df["val_acc"].max():.4f} ({eff_df["val_acc"].max()*100:.2f}%)')
print(f'SimpleCNN    best val_acc: {cnn_df["val_acc"].max():.4f} ({cnn_df["val_acc"].max()*100:.2f}%)')

---
## Next Steps

After this notebook finishes:
1. The improved EfficientNet weights are saved to `backend/model/weights/efficientnet_b0_finetuned.pth` (overwritten with better weights)
2. Run `docker-compose up --build` to rebuild the backend with the new weights
3. Run `03_evaluation.ipynb` to regenerate the confusion matrix, classification report, and ROC curve with the improved model
4. The SimpleCNN weights are at `backend/model/weights/custom_cnn_baseline.pth` — use them in your report as a baseline comparison